In [1]:
import pandas as pd
import numpy as np
import os, re

In [2]:

# ===== USER INPUT =====
monomer_tsv  = "/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/reference_pdbs/mapping_monomers_BSA/sasa_per_residue_monomer.tsv"   # from ChimeraX script
capsid_tsv   = "/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/reference_pdbs/mapping_monomers_BSA/sasa_per_residue_capsid.tsv"    # from ChimeraX script
contacts_tsv = "/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/reference_pdbs/new_comparisons/RF/mapped_interactions_RF.tsv"                           # your MSA dataframe
all_sasa_tsv = "/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/reference_pdbs/mapping_monomers_BSA/sasa_per_residue.tsv"                  # all proteins monomer SASA
output_dir   = "./"

In [3]:
# ===== HELPER: extract shared accession from both naming formats ==============
def extract_imgvr_id(name):
    """
    Handles two accession families:

    IMGVR style:
      - sasa:     IMGVR_UViG_3300000213_000001_3300000213_LP_F_10_...
      - contacts: IMGVR_UViG_3300013372_000845|3300013372|...
      → returns 'IMGVR_UViG_3300000213_000001'

    MGYP style:
      - sasa:     MGYP000117239797_CR_1_FL_1_unrelaxed_rank_001_...
      - contacts: MGYP000117239797
      → returns 'MGYP000117239797'
    """
    # IMGVR: four underscore-delimited tokens
    m = re.match(r"(IMGVR_UViG_\d+_\d+)", str(name))
    if m:
        return m.group(1)

    # MGYP: alphanumeric accession before first underscore
    m = re.match(r"(MGYP\d+)", str(name))
    if m:
        return m.group(1)

    return None


In [4]:

# ===== STEP 1: Build EC6098 reduction map =====================================
print("── Step 1: Building EC6098 reduction map ────────────────────────────────")

monomer_sasa = pd.read_csv(monomer_tsv, sep="\t")
capsid_sasa  = pd.read_csv(capsid_tsv,  sep="\t")

mono = monomer_sasa[["position", "AA", "sasa"]].rename(columns={"sasa": "sasa_monomer"})
caps = capsid_sasa[["position", "sasa"]].rename(columns={"sasa": "sasa_capsid"})

ec6098_map = mono.merge(caps, on="position", how="inner")
ec6098_map["reduction_factor"] = (
    ec6098_map["sasa_capsid"] / ec6098_map["sasa_monomer"]
).clip(upper=1.0)
ec6098_map.loc[ec6098_map["sasa_monomer"] == 0, "reduction_factor"] = np.nan

ec6098_map_path = os.path.join(output_dir, "ec6098_sasa_reduction_map.tsv")
ec6098_map.to_csv(ec6098_map_path, sep="\t", index=False)
print(f"  Reduction map saved: {ec6098_map_path}")
print(f"  Residues mapped: {len(ec6098_map)}, valid factor: {ec6098_map['reduction_factor'].notna().sum()}")

reduction_lookup = ec6098_map.set_index("position")["reduction_factor"].to_dict()


── Step 1: Building EC6098 reduction map ────────────────────────────────
  Reduction map saved: ./ec6098_sasa_reduction_map.tsv
  Residues mapped: 485, valid factor: 470


In [5]:
# ===== STEP 2: Build name concordance table ===================================
print("\n── Step 2: Building name concordance ────────────────────────────────────")

sasa_df  = pd.read_csv(all_sasa_tsv, sep="\t")
contacts = pd.read_csv(contacts_tsv, sep="\t")

sasa_proteins     = sasa_df["protein"].unique()
contacts_proteins = contacts["protein"].unique()

sasa_ids     = {p: extract_imgvr_id(p) for p in sasa_proteins}
contacts_ids = {p: extract_imgvr_id(p) for p in contacts_proteins}

# Invert: accession → contacts-style name
accession_to_contacts = {v: k for k, v in contacts_ids.items() if v is not None}

concordance = []
for sasa_name, accession in sasa_ids.items():
    contacts_name = accession_to_contacts.get(accession)
    concordance.append({
        "sasa_name":     sasa_name,
        "accession":     accession,
        "contacts_name": contacts_name
    })

concordance_df = pd.DataFrame(concordance)
n_matched = concordance_df["contacts_name"].notna().sum()
print(f"  SASA proteins:             {len(sasa_proteins)}")
print(f"  Contacts proteins:         {len(contacts_proteins)}")
print(f"  Successfully matched:      {n_matched}")
print(f"  Unmatched (no contacts):   {concordance_df['contacts_name'].isna().sum()}")

concordance_df.to_csv(os.path.join(output_dir, "name_concordance.tsv"), sep="\t", index=False)

sasa_to_contacts = concordance_df.dropna(subset=["contacts_name"]) \
                                  .set_index("sasa_name")["contacts_name"].to_dict()


── Step 2: Building name concordance ────────────────────────────────────
  SASA proteins:             1111
  Contacts proteins:         1111
  Successfully matched:      1110
  Unmatched (no contacts):   1


In [6]:
# ===== STEP 3: Build MSA position maps =======================================
print("\n── Step 3: Building MSA position maps ───────────────────────────────────")

ec6098_contacts = contacts[contacts["protein"] == "EC6098_reference"].copy()

msa_col_to_ec6098_pos = {}
msa_col_to_ec6098_pos.update(
    ec6098_contacts[["A_msa_col", "A_ref_pos"]].drop_duplicates()
    .set_index("A_msa_col")["A_ref_pos"].to_dict()
)
msa_col_to_ec6098_pos.update(
    ec6098_contacts[["B_msa_col", "B_ref_pos"]].drop_duplicates()
    .set_index("B_msa_col")["B_ref_pos"].to_dict()
)

protein_msa_map = (
    contacts[["protein", "A_msa_col", "A_pos"]]
    .drop_duplicates()
    .rename(columns={"A_msa_col": "msa_col", "A_pos": "local_pos"})
    .copy()
)

# Coerce to numeric — '-' gap characters become NaN and are dropped
protein_msa_map["local_pos"] = pd.to_numeric(protein_msa_map["local_pos"], errors="coerce")
protein_msa_map["msa_col"]   = pd.to_numeric(protein_msa_map["msa_col"],   errors="coerce")
protein_msa_map = protein_msa_map.dropna(subset=["local_pos", "msa_col"])
protein_msa_map["local_pos"] = protein_msa_map["local_pos"].astype(int)
protein_msa_map["msa_col"]   = protein_msa_map["msa_col"].astype(int)

# Same cleanup for the EC6098 lookup keys
msa_col_to_ec6098_pos = {
    k: v for k, v in msa_col_to_ec6098_pos.items()
    if pd.to_numeric(k, errors="coerce") is not np.nan
}
msa_col_to_ec6098_pos = {
    int(k): int(v) for k, v in msa_col_to_ec6098_pos.items()
}
reduction_lookup = {int(k): v for k, v in reduction_lookup.items()}

print(f"  MSA columns with EC6098 mapping: {len(msa_col_to_ec6098_pos)}")
print(f"  Rows in protein_msa_map after gap removal: {len(protein_msa_map)}")


── Step 3: Building MSA position maps ───────────────────────────────────
  MSA columns with EC6098 mapping: 475
  Rows in protein_msa_map after gap removal: 492131


In [11]:
protein_msa_map

,protein,msa_col,local_pos
0,EC6098_reference,561,562
1,EC6098_reference,563,564
2,EC6098_reference,360,361
10,EC6098_reference,560,561
23,EC6098_reference,490,491
...,...,...,...
48845808,MGYP003333944615,113,100
48852595,MGYP003333944615,220,205
48853196,MGYP003333944615,386,342
48854314,MGYP003333944615,184,169


In [13]:
protein_msa_map.loc[protein_msa_map['protein'].str.contains("IMGVR_UViG_3300027969_000035")].loc[protein_msa_map['msa_col'] == 555]

,protein,msa_col,local_pos
50348,IMGVR_UViG_3300027969_000035|3300027969|Ga0209...,555,512


In [14]:

# ===== STEP 4: Apply reduction factors =======================================
print("\n── Step 4: Applying reduction factors ───────────────────────────────────")

results = []
skipped_proteins = []

for sasa_protein, group in sasa_df.groupby("protein"):

    # Translate sasa-style name → contacts-style name
    contacts_protein = sasa_to_contacts.get(sasa_protein)
    if contacts_protein is None:
        skipped_proteins.append(sasa_protein)
        # Still write rows but with NaN factors
        for _, row in group.iterrows():
            results.append({
                "protein":          sasa_protein,
                "accession":        sasa_ids.get(sasa_protein),
                "local_pos":        row["position"],
                "AA":               row["AA"],
                "sasa_monomer":     row["sasa"],
                "msa_col":          None,
                "ec6098_pos":       None,
                "reduction_factor": np.nan,
                "sasa_capsid_est":  np.nan
            })
        continue

    prot_map = protein_msa_map[protein_msa_map["protein"] == contacts_protein]
    local_to_msa = prot_map.set_index("local_pos")["msa_col"].to_dict()

    for _, row in group.iterrows():
        local_pos = row["position"]
        sasa      = row["sasa"]
        aa        = row["AA"]

        msa_col    = local_to_msa.get(local_pos)
        ec6098_pos = msa_col_to_ec6098_pos.get(msa_col)
        factor     = reduction_lookup.get(ec6098_pos, np.nan)

        results.append({
            "protein":          sasa_protein,
            "accession":        sasa_ids.get(sasa_protein),
            "local_pos":        local_pos,
            "AA":               aa,
            "sasa_monomer":     sasa,
            "msa_col":          msa_col,
            "ec6098_pos":       ec6098_pos,
            "reduction_factor": factor,
            "sasa_capsid_est":  sasa * factor if not np.isnan(factor) else np.nan
        })

output = pd.DataFrame(results)

n_total    = len(output)
n_mapped   = output["reduction_factor"].notna().sum()
n_unmapped = output["reduction_factor"].isna().sum()

print(f"  Total residues:                   {n_total}")
print(f"  With reduction factor:            {n_mapped}  ({100*n_mapped/n_total:.1f}%)")
print(f"  Without factor (not in MSA map):  {n_unmapped} ({100*n_unmapped/n_total:.1f}%)")
print(f"  Proteins with no contacts match:  {len(skipped_proteins)}")
if skipped_proteins:
    print("  Examples:", skipped_proteins[:3])

output_path = os.path.join(output_dir, "sasa_capsid_estimated_all_proteins.tsv")
# output.to_csv(output_path, sep="\t", index=False)

# concordance_df.to_csv(os.path.join(output_dir, "name_concordance.tsv"), sep="\t", index=False)
print(f"\n  Saved: {output_path}")
print("All done!")


── Step 4: Applying reduction factors ───────────────────────────────────
  Total residues:                   572053
  With reduction factor:            475298  (83.1%)
  Without factor (not in MSA map):  96755 (16.9%)
  Proteins with no contacts match:  1
  Examples: ['EC6098_reference_unrelaxed_rank_001_alphafold2_ptm_model_3_seed_000']

  Saved: ./sasa_capsid_estimated_all_proteins.tsv
All done!
